# E1 · Optimizing and evaluating on the same data

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Find out why a random forest tuned with `GridSearchCV` "gets" an AUC of 0.95 on the test set on Titanic, fix the
scheme (split, search on the training set only, test set touched once) and measure how much the number was inflated.

## Dataset

`ds.titanic()` with the usual 75/25 split: `X_tr, X_te, y_tr, y_te`. `X, y` are **all** the data (training + test). The preprocessor
and the pipeline with the random forest are in the colleague's cell.

## Educational purpose

Recognize the most common leakage scheme in tuning: after splitting the data, the search is run on the whole dataset "because
it needs more data" and then you evaluate on a part that the search has already seen.

## Task

1. Run the colleague's cell: `auc_bug` is very high (about 0.95) and well above the `best_score_` of its own search (about 0.87).
2. Compute `test_share_in_training`: the fraction of rows of `X_te` that also appear in the `X` the search trained on (index). Is it 1?
3. Fix it: run the search **only** on `X_tr, y_tr` and evaluate once on `X_te`. Save `auc_ok` and `best_ok` (the `best_score_` of the correct search).
4. Save `inflation = auc_bug − auc_ok`: by how much the colleague's method overestimates the performance.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
NUM = ["age", "siblings_spouses", "parents_children", "fare"]
CAT = ["sex", "port", "deck", "title"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)

prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), CAT),
    ("pclass", "passthrough", ["pclass"]),
], sparse_threshold=0)
pipe = make_pipeline(prep, RandomForestClassifier(random_state=0))
grid = {"randomforestclassifier__max_depth": [4, 8, None], "randomforestclassifier__min_samples_leaf": [1, 3]}

## Your work

### The colleague's code

In [ ]:
bug_search = GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc").fit(X, y)             # X, y: ALL the data
auc_bug = roc_auc_score(y_te, bug_search.predict_proba(X_te)[:, 1])
print(f"test set AUC: {auc_bug:.3f}   (best_score_ of the search: {bug_search.best_score_:.3f})")

### Your fix

In [ ]:
test_share_in_training = None
auc_ok = auc_bug
best_ok = bug_search.best_score_
inflation = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`ok_search = GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc").fit(X_tr, y_tr)`; then `roc_auc_score(y_te, ok_search.predict_proba(X_te)[:, 1])` and `ok_search.best_score_`.'):
    check(auc_bug > 0.93, "the colleague's number is too good for the Titanic: it is leakage", "auc_bug should exceed 0.93; don't modify the colleague's code")
    check(test_share_in_training == 1.0, "all the rows of the test set were in the search", "test_share_in_training = X_te.index.isin(X.index).mean()")
    check(auc_ok < 0.92, "with the search on the training set only the AUC is realistic (below 0.92)", "the search must see only X_tr, y_tr")
    check(close(inflation, auc_bug - auc_ok) and inflation > 0.05, "the colleague overestimated the performance by more than 0.05 of AUC",
          "inflation = auc_bug - auc_ok")
    check(abs(best_ok - auc_ok) < 0.05, "best_score_ (cross-validation) and the correct test AUC are close", "best_ok = best_score_ of the search on the training set alone")
done()

---
The commented solution is in `solutions/E_find_the_bug/E1_search_and_evaluation_on_the_same_data.ipynb`. Open it after you pass the check, or after you have really tried.